# FoodGuard AI — Passport & Certificate Generation

**Purpose**: Generate investigation passport with full traceability.

This notebook:
1. Assembles comprehensive investigation report (JSON)
2. Generates Markdown version for readability
3. Computes SHA-256 hash of passport JSON
4. Generates QR code (hash encoded)
5. Stores passport & certificate in DB
6. Saves QR code image to disk

---

**Run this AFTER notebook 06 (LLM reasoning).**

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from datetime import datetime
from pathlib import Path

from foodguard_lib import (
    DB_PATH,
    get_investigation,
    get_aroma_analysis,
    get_taste_analysis,
    get_vision_analysis,
    execute_query,
    compute_sha256,
    generate_qr_code,
    generate_report_markdown,
    generate_passport_id,
    dict_from_row,
    ensure_directories
)

ensure_directories()
print("[OK] Imports successful")

## 2. Define Passport Generation Function

In [ ]:
def generate_investigation_passport(
    investigation_id: str,
    db_path: str = DB_PATH,
    cert_dir: str = "../certificates"
) -> dict:
    """
    Generate full investigation passport.

    Returns dict with:
      - passport_json: Full investigation data
      - passport_hash: SHA-256 hash
      - qr_code_path: Path to QR code image
      - passport_id: Unique passport identifier
    """

    # Fetch investigation
    investigation = get_investigation(investigation_id, db_path=db_path)
    if not investigation:
        raise ValueError(f"Investigation {investigation_id} not found")

    batch_id = investigation.get("batch_id")

    # Fetch all evidence
    aroma = get_aroma_analysis(batch_id, db_path=db_path) or {}
    taste = get_taste_analysis(batch_id, db_path=db_path) or {}
    vision = get_vision_analysis(batch_id, db_path=db_path) or {}

    # Fetch agent executions
    exec_query = """SELECT * FROM agent_execution WHERE investigation_id = ?
                      ORDER BY created_at ASC"""
    exec_rows = execute_query(db_path, exec_query, (investigation_id,))
    agent_executions = [dict_from_row(row) for row in exec_rows]

    # Build passport JSON
    passport_json = {
        "passport_id": generate_passport_id(),
        "investigation_id": investigation_id,
        "batch_id": batch_id,
        "timestamp_generated": datetime.utcnow().isoformat(),
        "verdict": {
            "suspected_adulterant": investigation.get("predicted_class"),
            "confidence": investigation.get("confidence"),
            "risk_level": investigation.get("risk_level"),
            "reasoning": investigation.get("reasoning")
        },
        "evidence": {
            "aroma": {
                "sensors": {
                    "ammonia": aroma.get("ammonia"),
                    "alcohol": aroma.get("alcohol"),
                    "voc": aroma.get("voc"),
                    "sulfur": aroma.get("sulfur"),
                    "hydrocarbon": aroma.get("hydrocarbon")
                },
                "predicted_class": aroma.get("predicted_class"),
                "confidence": aroma.get("confidence")
            },
            "taste": {
                "sensors": {
                    "sweetness": taste.get("sweetness"),
                    "saltiness": taste.get("saltiness"),
                    "sourness": taste.get("sourness"),
                    "bitterness": taste.get("bitterness"),
                    "umami": taste.get("umami"),
                    "astringency": taste.get("astringency")
                },
                "predicted_class": taste.get("predicted_class"),
                "confidence": taste.get("confidence")
            },
            "vision": {
                "image_path": vision.get("image_path"),
                "deposit_type": vision.get("deposit_type"),
                "predicted_class": vision.get("predicted_class"),
                "confidence": vision.get("confidence")
            }
        },
        "agent_executions": [
            {
                "agent_name": ae.get("agent_name"),
                "reasoning": ae.get("reasoning"),
                "execution_time_ms": ae.get("execution_time_ms"),
                "timestamp": ae.get("created_at")
            }
            for ae in agent_executions
        ]
    }

    # Compute hash
    passport_hash = compute_sha256(passport_json)

    # Generate QR code
    qr_filename = f"{investigation_id}_passport.png"
    qr_path = str(Path(cert_dir) / qr_filename)
    generate_qr_code(passport_hash, output_path=qr_path)

    return {
        "passport_json": passport_json,
        "passport_hash": passport_hash,
        "qr_code_path": qr_path,
        "passport_id": passport_json["passport_id"]
    }

print("[OK] Passport generation function defined")

## 3. Get Test Investigation

In [ ]:
# Get an investigation
query = "SELECT id FROM investigations WHERE risk_level IS NOT NULL LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    test_investigation_id = dict_from_row(rows[0])["id"]
    print(f"Test Investigation ID: {test_investigation_id}")
else:
    print("[ERROR] No completed investigations found. Run notebook 06 first.")
    test_investigation_id = None

## 4. Generate Passport

In [ ]:
if test_investigation_id:
    print(f"Generating passport for {test_investigation_id}...\n")

    passport_data = generate_investigation_passport(test_investigation_id, db_path=DB_PATH)

    print("="*60)
    print("INVESTIGATION PASSPORT")
    print("="*60)
    print(f"\nPassport ID: {passport_data['passport_id']}")
    print(f"Investigation ID: {test_investigation_id}")
    print(f"\nVerdict:")
    verdict = passport_data["passport_json"]["verdict"]
    print(f"  Suspected Adulterant: {verdict['suspected_adulterant']}")
    print(f"  Confidence: {verdict['confidence']:.1%}")
    print(f"  Risk Level: {verdict['risk_level']}")
    print(f"\nPassport Hash (SHA-256):")
    print(f"  {passport_data['passport_hash']}")
    print(f"\nQR Code Path:")
    print(f"  {passport_data['qr_code_path']}")
    print(f"\n{'='*60}")

## 5. Store Passport in DB

In [ ]:
if test_investigation_id:
    # Insert passport record
    insert_query = """INSERT INTO passports
        (id, investigation_id, passport_json, hash_sha256, qr_code_path, blockchain_status)
        VALUES (?, ?, ?, ?, ?, ?)"""

    from foodguard_lib import execute_insert

    passport_id = passport_data["passport_id"]
    execute_insert(
        DB_PATH,
        insert_query,
        (
            passport_id,
            test_investigation_id,
            json.dumps(passport_data["passport_json"]),
            passport_data["passport_hash"],
            passport_data["qr_code_path"],
            "pending"
        )
    )

    print(f"[OK] Passport stored in DB with ID: {passport_id}")

## 6. Generate Markdown Report

In [ ]:
if test_investigation_id:
    # Get investigation details
    investigation = get_investigation(test_investigation_id, db_path=DB_PATH)

    # Generate markdown report
    report_md = generate_report_markdown(
        investigation_id=test_investigation_id,
        suspected_adulterant=investigation.get("predicted_class"),
        confidence=investigation.get("confidence", 0.0),
        risk_level=investigation.get("risk_level"),
        reasoning=investigation.get("reasoning", "No reasoning available"),
        aroma_conf=get_aroma_analysis(investigation.get("batch_id"), db_path=DB_PATH).get("confidence", 0.0),
        taste_conf=get_taste_analysis(investigation.get("batch_id"), db_path=DB_PATH).get("confidence", 0.0),
        vision_conf=get_vision_analysis(investigation.get("batch_id"), db_path=DB_PATH).get("confidence", 0.0),
        matched_rules=[]
    )

    print("\n" + "="*60)
    print("MARKDOWN REPORT PREVIEW")
    print("="*60)
    print(report_md[:1000])  # First 1000 chars
    print("\n[...truncated...]")

## 7. Test: Generate Multiple Passports

In [ ]:
# Get completed investigations and generate passports for each
query = """SELECT id FROM investigations
           WHERE risk_level IS NOT NULL
           ORDER BY created_at DESC LIMIT 5"""
rows = execute_query(DB_PATH, query)
investigation_ids = [dict_from_row(row)["id"] for row in rows]

print(f"Generating passports for {len(investigation_ids)} investigations...\n")

passport_count = 0
for i, inv_id in enumerate(investigation_ids, 1):
    try:
        passport_data = generate_investigation_passport(inv_id, db_path=DB_PATH)

        # Store in DB
        insert_query = """INSERT INTO passports
            (id, investigation_id, passport_json, hash_sha256, qr_code_path, blockchain_status)
            VALUES (?, ?, ?, ?, ?, ?)"""

        from foodguard_lib import execute_insert
        execute_insert(
            DB_PATH,
            insert_query,
            (
                passport_data["passport_id"],
                inv_id,
                json.dumps(passport_data["passport_json"]),
                passport_data["passport_hash"],
                passport_data["qr_code_path"],
                "pending"
            )
        )

        print(f"{i}. {inv_id}")
        print(f"   Passport: {passport_data['passport_id']}")
        print(f"   Hash: {passport_data['passport_hash'][:16]}...")
        print(f"   QR Code: {Path(passport_data['qr_code_path']).name}")
        passport_count += 1
    except Exception as e:
        print(f"{i}. {inv_id} → ERROR: {e}")

print(f"\n[OK] Generated {passport_count} passports")

## 8. Verify Passports in DB

In [ ]:
# Check passports table
query = "SELECT id, investigation_id, hash_sha256 FROM passports ORDER BY created_at DESC LIMIT 5"
rows = execute_query(DB_PATH, query)

print(f"Passports in DB: {len(rows)}\n")
for i, row in enumerate(rows, 1):
    passport_dict = dict_from_row(row)
    print(f"{i}. Passport: {passport_dict['id']}")
    print(f"   Investigation: {passport_dict['investigation_id']}")
    print(f"   Hash: {passport_dict['hash_sha256'][:32]}...")
    print()

## 9. Display QR Code Example

In [ ]:
# Show one of the generated QR codes
if rows:
    first_passport = dict_from_row(rows[0])
    passport_json_str = "SELECT qr_code_path FROM passports LIMIT 1"
    qr_rows = execute_query(DB_PATH, passport_json_str)
    if qr_rows:
        qr_path = dict_from_row(qr_rows[0]).get("qr_code_path")
        print(f"Generated QR code at: {qr_path}")

        from pathlib import Path
        if Path(qr_path).exists():
            print("[OK] QR code file exists")
            # Display the image
            from PIL import Image
            img = Image.open(qr_path)
            print(f"QR code size: {img.size}")
        else:
            print("[WARNING] QR code file not found")

## ✅ Passport & Certificate Generation Complete!

**Summary**:
- ✓ Comprehensive investigation passports generated (JSON)
- ✓ SHA-256 hashes computed for integrity
- ✓ QR codes generated and saved to disk
- ✓ Passports stored in DB with metadata
- ✓ Markdown reports generated for readability
- ✓ Full traceability with all agent executions

**Next Steps**:
1. Run `08_gradio_demo.ipynb` for full end-to-end interactive demo